# Jour 1 · Découper et préparer sans tricher

## Objectifs

- comprendre les rôles du train, de la validation et du test
- utiliser un découpage stratifié et reproductible
- éviter les fuites pendant la préparation


## Pourquoi trois ensembles ?

Si nous évaluons un modèle sur les mêmes exemples que ceux utilisés pour apprendre, nous mesurons surtout sa mémoire. Le **train** ajuste le modèle. La **validation** aide à comparer des choix. Le **test** reste fermé jusqu'à la décision finale et simule des données inconnues.


![Rôles du train de la validation et du test](../ressources/illustrations/jour_01/02_train_validation_test.png)

*Le test ne participe ni à l’apprentissage ni au choix du modèle.*


## Conserver la classe rare dans chaque ensemble

![Illustration du découpage stratifié](../ressources/illustrations/jour_01/02_stratification.png)

Dans ce cours, environ 60 % des lignes servent au **train**, 20 % à la **validation** et 20 % au **test**. Ces proportions ne sont pas universelles : elles doivent laisser assez d’exemples pour apprendre et assez d’exemples indépendants pour mesurer.

Comme les pannes sont rares, un tirage entièrement aléatoire pourrait créer un petit ensemble contenant trop peu de pannes. `stratify=y` demande de conserver approximativement la proportion de chaque classe.

Stratifier ne fabrique pas de nouvelles pannes et ne rend pas les classes équilibrées. Cela évite seulement qu’un ensemble reçoive une répartition très différente par hasard.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_course_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "ressources" / "donnees" / "maintenance_machines.csv").exists():
            return candidate
    raise FileNotFoundError("Racine du cours introuvable. Ouvrez le notebook depuis le dossier du cours.")


ROOT = find_course_root()
DATA_FILE = ROOT / "ressources" / "donnees" / "maintenance_machines.csv"
plt.style.use("seaborn-v0_8-whitegrid")

from sklearn.model_selection import train_test_split

df = pd.read_csv(DATA_FILE)
features = ["temperature_c", "vibration_mm_s", "pressure_bar", "load_pct", "age_years", "days_since_maintenance"]
X = df[features]
y = df["failure_7d"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=42
)
X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)
print("train      :", X_train.shape, f"pannes {y_train.mean():.1%}")
print("validation :", X_validation.shape, f"pannes {y_validation.mean():.1%}")
print("test       :", X_test.shape, f"pannes {y_test.mean():.1%}")


## Lire le code et les résultats du découpage

Le premier `train_test_split` garde 60 % des lignes dans le train et place 40 % dans `X_temp`. Le second partage ce bloc temporaire en deux moitiés : validation et test représentent donc chacun 20 % du total.

Les objets `X_train` et `y_train` doivent toujours rester alignés : la ligne d’une machine dans `X_train` correspond au label situé au même index dans `y_train`.

`random_state=42` fixe le tirage pseudo-aléatoire. Deux personnes obtiennent ainsi le même découpage et peuvent comparer leurs résultats. La valeur 42 n’a aucune propriété spéciale et n’améliore pas le modèle.

Après l’exécution, comparez les formes et les taux de panne affichés. Ils ne sont pas rigoureusement identiques, mais doivent rester proches grâce à la stratification.

## La fuite de données

Une fuite apparaît lorsqu’une information venant de la validation, du test ou du futur influence l’apprentissage. Le score obtenu paraît alors meilleur que ce que le modèle pourra réellement reproduire.


![Mauvaise et bonne préparation des données](../ressources/illustrations/jour_01/02_fuite_donnees.png)

*On découpe d’abord ; les transformations apprennent ensuite leurs paramètres uniquement sur le train.*


### Trois fuites fréquentes

- **Fuite de cible** : la réponse, ou une colonne qui la révèle presque directement, se retrouve dans `X`.
- **Fuite de préparation** : une moyenne, un minimum ou une sélection de colonnes est calculé sur tout le jeu avant le découpage.
- **Fuite temporelle** : une information connue après l’instant de décision sert à prédire le passé.

Par exemple, le nombre de jours nécessaires pour réparer une panne ne peut pas servir à prévoir cette panne : cette durée n’est connue qu’après l’événement.

La règle pratique est simple : se placer mentalement à l’instant de la prédiction et masquer tout ce qui ne serait pas encore connu.


## Pourquoi utiliser une Pipeline ?

Certaines méthodes fonctionnent mieux lorsque les variables sont standardisées. `StandardScaler` apprend la moyenne et l'écart-type de chaque colonne. Une `Pipeline` garantit que le scaler est ajusté sur le train pendant `fit`, puis simplement réutilisé pendant `predict`.

`StandardScaler` réalise une **standardisation** : pour chaque colonne, il retire la moyenne du train puis divise par son écart-type. Les variables se retrouvent ainsi sur des échelles comparables. Il mémorise les valeurs du train et réutilise exactement les mêmes sur validation et test.

La **régression logistique**, malgré son nom, est ici un modèle de classification. Elle combine les variables pour produire un score de probabilité de la classe `1`. `max_iter=1000` lui laisse assez d'étapes pour terminer son ajustement. `class_weight="balanced"` accorde automatiquement davantage d'importance à la classe rare afin que les pannes ne soient pas noyées par les nombreux cas normaux.



![Standardisation et Pipeline sans fuite](../ressources/illustrations/jour_01/02_standardisation_pipeline.png)

### `fit`, `transform` et `predict`

Un transformateur possède deux étapes distinctes. `fit` apprend ce qui est nécessaire sur le train, ici la moyenne et l’écart-type de chaque colonne. `transform` applique ensuite exactement la même transformation au train, à la validation ou au test.

La standardisation ne supprime pas les valeurs extrêmes et ne rend pas une variable « normale ». Elle recentre simplement chaque colonne autour de zéro et ramène son échelle à une variation comparable. Elle aide souvent les modèles linéaires, les méthodes basées sur des distances et les réseaux de neurones. Les arbres de décision y sont généralement moins sensibles.

Une `Pipeline` enchaîne les opérations dans le bon ordre. Lors de `fit`, elle ajuste le scaler puis le modèle sur le train. Lors de `predict`, elle transforme les nouvelles lignes avec les valeurs déjà apprises avant de les transmettre au modèle.

### Préparer ne signifie pas seulement standardiser

Selon le jeu de données, il faut aussi traiter les valeurs manquantes, les catégories, les doublons, les unités incohérentes et les valeurs impossibles. Toute opération dont le résultat dépend des données doit être apprise sur le train uniquement et, idéalement, placée dans la Pipeline.


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
)
pipeline.fit(X_train, y_train)
print("Pipeline entraînée uniquement sur le train.")


## À vous de jouer — vérifier le découpage

Calculez la proportion de lignes et le taux de panne de chacun des trois ensembles dans un petit DataFrame.


In [ ]:
# Créez un DataFrame avec les colonnes lignes, proportion et taux_panne.
pass


**À retenir :** toute opération qui « apprend » quelque chose des données doit être ajustée sur le train seulement.
